# Cross-Border & FX

Build a **grounded, self-verifying** agent for a payments processor's **cross-border book** — settlement volume, FX rate, FX margin (bps) and **FX PnL** by corridor (origin→dest / currency pair).

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`CROSS_BORDER_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Data note:** this lab is **fully synthetic** — corridor volume *and* the FX series. Real FX / exchange-rate data is **not free** in this account; it is a **paid Snowflake Marketplace add-on** (e.g. Cybersyn FX listings). See `CROSS_BORDER_DATA_SOURCE_INFO` for the go-live path.

**Prereq:** run `setup_synthetic.sql` (TRY) or `setup.sql` (DEPLOY — still synthetic here). Set the notebook role to one that owns `SNOWPACK_CROSS_BORDER_FX.CORE`.

In [ ]:
SCHEMA = "SNOWPACK_CROSS_BORDER_FX.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.CROSS_BORDER_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "\u2014", mode[1])

## Step 1 — Explore the benchmark

`CROSS_BORDER_BENCHMARK` = cross-border volume, FX rate, FX margin (bps) and FX PnL by corridor/month (100 bps = 1%).

In [ ]:
session.sql(f"""
  SELECT corridor_id, currency_pair, month_end,
         ROUND(cross_border_volume_usd/1e6,1) AS volume_m,
         fx_rate, fx_margin_bps,
         ROUND(fx_pnl_usd/1e3,1) AS fx_pnl_k
  FROM {SCHEMA}.CROSS_BORDER_BENCHMARK
  WHERE month_end = (SELECT MAX(month_end) FROM {SCHEMA}.CROSS_BORDER_BENCHMARK)
  ORDER BY fx_pnl_usd DESC
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`CROSS_BORDER_SV` lets Analyst answer NL questions like *"which corridors made the most FX PnL in 2026?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English.

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.CROSS_BORDER_SV
      METRICS total_fx_pnl, total_cross_border_volume, avg_fx_margin_bps
      DIMENSIONS currency_pair, report_year)
  WHERE report_year = 2026 ORDER BY total_fx_pnl DESC
""").to_pandas()

## Step 3 — Verified fact pack

Start-of-window and latest snapshots per corridor — the agent's only allowed numbers.

In [ ]:
fact_pack = session.sql(f"""
  WITH w AS (SELECT MIN(month_end) a, MAX(month_end) z FROM {SCHEMA}.CROSS_BORDER_BENCHMARK)
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'corridor', s.corridor_id, 'currency_pair', s.currency_pair,
           'month', s.month_end::string,
           'cross_border_volume_usd', s.cross_border_volume_usd,
           'fx_rate', s.fx_rate, 'fx_margin_bps', s.fx_margin_bps,
           'fx_pnl_usd', s.fx_pnl_usd))
  FROM {SCHEMA}.CROSS_BORDER_BENCHMARK s JOIN w ON s.month_end IN (w.a, w.z)
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a cross-border payments analyst. Write a concise corridor / FX note using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; fx_margin_bps is basis points (100 bps = 1%); dollar fields are USD, express in $B/$M; "
    "compute volume and FX PnL change as (latest - earliest) per corridor from FACTS; FX rates are SYNTHETIC (real FX is a paid add-on); "
    "no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (which corridors grew volume / FX PnL), a table "
    "(corridor | currency pair | volume start->now | fx margin bps | fx PnL start->now), then 3 bullets, ending with '## Data Coverage' (source + window + synthetic FX note)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.CROSS_BORDER_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Cross-border FX performance", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.CROSS_BORDER_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified cross-border / FX loop on fully synthetic data. `setup.sql` already created a **`CROSS_BORDER_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which corridors made the most FX PnL in 2026?"*).

**Go live:** subscribe to a **paid** Marketplace FX listing (e.g. Cybersyn FX / exchange-rate), replace `CROSS_BORDER_FX_MONTHLY` with a view over it (`currency_pair, month_end, fx_rate`), and point `CROSS_BORDER_VOLUME_MONTHLY` at your settlement data. Everything downstream is unchanged. This grounded-plus-verifier pattern is reusable across payments analytics.

**Try breaking the verifier:** add a made-up claim (e.g., *"USD/MXN will likely strengthen next quarter"*) to the draft and watch Step 5 flag it *Needs review*.